<a href="https://colab.research.google.com/github/sherf1207/AI-Projects/blob/main/Word2Vec%20on%20Restaurant%20Reviews%3A%20CBOW%20vs%20Skip-Gram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project implements Word2Vec using CBOW and Skip-Gram on restaurant reviews to learn word embeddings from text. The CBOW model achieved 82% accuracy, performing well on the small dataset, while Skip-Gram achieved only 8% accuracy due to limited data and lack of negative sampling. The results highlight that CBOW is more effective than Skip-Gram in low-data scenarios.

In [10]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

df = pd.read_csv('/content/Restaurant_Reviews.tsv', sep='\t')
df=df.drop(columns='Liked')
df

,Review
0,Wow... Loved this place.
1,Crust is not good.
2,Not tasty and the texture was just nasty.
3,Stopped by during the late May bank holiday of...
4,The selection on the menu was great and so wer...
...,...
995,I think food should have flavor and texture an...
996,Appetite instantly gone.
997,Overall I was not impressed and would not go b...
998,"The whole experience was underwhelming, and I ..."


#Applying Cbow on restaurant-reviews dataset

In [11]:
def clean_text(text):
    text= text.lower()
    return re.sub(r"[^a-zA-Z0-9\s]", "", text)

df['Review']= df['Review'].apply(clean_text)

tokenizer=Tokenizer()
tokenizer.fit_on_texts(df['Review'])
sequences= tokenizer.texts_to_sequences(df['Review'])
vocab_size= len(tokenizer.word_index)+1

windows=2
x_CBow= []
y_CBow= []

for seq in sequences:
    for i in range(windows, len(seq)-windows):
        context= seq[i-windows : i] + seq[i+1 : i+windows+1]
        target= seq[i]
        x_CBow.append(context)
        y_CBow.append(target)

x_CBow= np.array(x_CBow)
y_CBow= np.array(y_CBow)

print('There are ', vocab_size, ' words in this data')

There are  2080  words in this data


In [12]:
embedding_dimention=100
model=Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dimention, input_length=windows*2))
model.add(GlobalAveragePooling1D())
model.add(Dense(vocab_size,activation="softmax"))
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(x_CBow,y_CBow, epochs=40 ,verbose=1)

Epoch 1/40


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.0489 - loss: 7.2912
Epoch 2/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0586 - loss: 6.2761
Epoch 3/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0684 - loss: 6.0742
Epoch 4/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0714 - loss: 5.9476
Epoch 5/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.0823 - loss: 5.8273
Epoch 6/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.0900 - loss: 5.7042
Epoch 7/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0980 - loss: 5.5757
Epoch 8/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.1093 - loss: 5.4396
Epoch 9/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.1221 - loss: 5.2961
Epoch 10/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.1337 - loss: 5.1461
Epoch 11/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.1507 - loss: 4.9900
Epoch 12/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accur

#Applying Skip-Gram on restaurant-reviews dataset

In [17]:
x_skipGram= []
y_skipGram= []

for seq in sequences:
    for i in range(windows, len(seq)-windows):
        target= seq[i]
        context= seq[i-windows : i]+ seq[i+1 : i+windows+1]

        for word in context:
            x_skipGram.append([target])
            y_skipGram.append(word)

x_skipGram= np.array(x_skipGram)
y_skipGram= np.array(y_skipGram)

In [18]:
SGmodel= Sequential()
SGmodel.add(Embedding(input_dim= vocab_size, output_dim= embedding_dimention, input_length=1, trainable=True))
SGmodel.add(GlobalAveragePooling1D())
SGmodel.add(Dense(vocab_size, activation = 'softmax'))
SGmodel.compile(optimizer= 'adam', loss= 'sparse_categorical_crossentropy', metrics= ['accuracy'])
SGmodel.fit(x_skipGram, y_skipGram, epochs = 20, verbose=1)

Epoch 1/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.0493 - loss: 6.8035
Epoch 2/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.0604 - loss: 6.0958
Epoch 3/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.0721 - loss: 5.9293
Epoch 4/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.0791 - loss: 5.7965
Epoch 5/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.0859 - loss: 5.6569
Epoch 6/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.0923 - loss: 5.5062
Epoch 7/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.0969 - loss: 5.3453
Epoch 8/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.1014 - loss: 5.1837
Epoch 9/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.1057 - loss: 5.0274
Epoch 10/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.1073 - loss: 4.8784
Epoch 11/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.1050 - loss: 4.7403
Epoch 12/20
869/869 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/ste